# Python Finance Paper Replication: Empirical Asset Pricing via ML, GKX

# Step 1: Let's build the 94 characteristics

## 1) Compustat/CRSP variables

According to GHZ SAS code and their literature.

### a) Annual Compustat Data

Get: sale, revt, cogs, xsga, dp, xrd, xad, ib, ebitda, ebit, nopi, spi, pi, txp, ni, txfed, txfo, txt, xint, capx, oancf, dvt, ob, gdwlia, gdwlip, gwo, rect, act, che, ppegt, invt, at, aco, intan, ao, ppent, gdwl, fatb, fatl, lct, dlc, dltt, lt, dm, dcvt, cshrc, dcpstk, pstk, ap, lco, lo, drc, drlt, txdi, ceq, scstkc, emp, csho, sic, naics, prcc_f, fyear, datadate, gvkey

### b) Quaterly Compustat Data

Get = ibq, saleq, txtq, revtq, cogsq, xsgaq, atq, actq, cheq, lctq, dlcq, ppentq, prccq, cshoq, seqq, pstkq, atq, ltq, pstkrq, fyearq, fqtr, rdq, sic, ceqq, gvkey, datadate

### c) Monthly CRSP Stock File

Get = permno, mthcaldt, mthret, mthprc, shrout, mthvol

### d) Daily CRSP Stock File

Get = permno, dlycaldt, dlyret, dlyprc, dlyvol, dlybid, dlyask, shrou

### e) Daily CRSP Market Index using SQL

Get = date, vwretd

### e) CRSP Delisting Information

Get = permno, delistingdt, delret, delretmisstype, delactiontype, delstatustype, delreasontype

### f) CRSP/Compustat Merged (CCM) link table

Get = gvkey, lpermno, lpermco, linktype, linkprim, linkdt, linkenddt

## 2) Use SQL to get the data

Will use SQL to pull data. Could use the Tidy Finance library (https://www.tidy-finance.org/chapters/wrds-crsp-and-compustat.html) which uses SQL too.

In [70]:
# Connect to WRDS

import pandas as pd
import numpy as np
import wrds

db = wrds.Connection(wrds_username='hcdb') 

Loading library list...
Done


In [2]:
# Get the Fundamentals Annual data using SQL

funda = db.raw_sql("""
    SELECT c.gvkey, c.cik, c.sic, c.naics,
           f.datadate, f.fyear, f.sale, f.revt, f.cogs, f.xsga, f.dp, f.xrd, f.xad, f.ib, f.ebitda,
           f.ebit, f.nopi, f.spi, f.pi, f.txp, f.ni, f.txfed, f.txfo, f.txt, f.xint,
           f.capx, f.oancf, f.dvt, f.ob, f.gdwlia, f.gdwlip, f.gwo,
           f.rect, f.act, f.che, f.ppegt, f.invt, f.at, f.aco, f.intan, f.ao,
           f.ppent, f.gdwl, f.fatb, f.fatl,
           f.lct, f.dlc, f.dltt, f.lt, f.dm, f.dcvt, f.cshrc, f.dcpstk, f.pstk,
           f.ap, f.lco, f.lo, f.drc, f.drlt, f.txdi,
           f.ceq, f.scstkc, f.emp, f.csho,
           f.prcc_f
    FROM comp.funda AS f
    JOIN comp.company AS c
      ON f.gvkey = c.gvkey
    WHERE f.datadate >= '1975-01-01'
      AND f.indfmt = 'INDL'
      AND f.datafmt = 'STD'
      AND f.popsrc = 'D'
      AND f.consol = 'C'
""").reset_index(drop=True)

funda_raw = funda.copy()

# select compustat variable from the fundamentals data (f) and the company reference (c)
# join them on gvkey
# use the correct data and the correct settings

In [3]:
display(funda.head(10))
display(list(funda.columns))
display(funda.shape)

,gvkey,cik,sic,naics,datadate,fyear,sale,revt,cogs,xsga,...,lco,lo,drc,drlt,txdi,ceq,scstkc,emp,csho,prcc_f
0,001000,<NA>,3089,<NA>,1975-12-31,1975,51.192,51.192,35.237,9.591,...,2.125,0.0,<NA>,<NA>,0.689,10.24,<NA>,1.054,2.098,4.375
1,001000,<NA>,3089,<NA>,1976-12-31,1976,66.414,66.414,45.277,12.396,...,2.485,0.0,<NA>,<NA>,0.237,14.106,<NA>,1.46,2.207,5.75
2,001000,<NA>,3089,<NA>,1977-12-31,1977,77.946,77.946,56.959,14.977,...,3.691,0.0,<NA>,<NA>,0.17,15.876,<NA>,1.741,2.226,9.25
3,001001,0000723576,5812,722,1978-12-31,1978,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,001001,0000723576,5812,722,1979-12-31,1979,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,001001,0000723576,5812,722,1980-12-31,1980,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,001001,0000723576,5812,722,1981-12-31,1981,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,001001,0000723576,5812,722,1982-12-31,1982,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,001001,0000723576,5812,722,1983-12-31,1983,25.395,25.395,6.31,16.435,...,0.706,0.0,<NA>,<NA>,0.144,7.823,<NA>,1.386,3.568,7.25
9,001001,0000723576,5812,722,1984-12-31,1984,32.007,32.007,8.171,20.628,...,0.788,0.0,<NA>,<NA>,0.275,8.962,<NA>,1.502,3.568,3.75


['gvkey',
 'cik',
 'sic',
 'naics',
 'datadate',
 'fyear',
 'sale',
 'revt',
 'cogs',
 'xsga',
 'dp',
 'xrd',
 'xad',
 'ib',
 'ebitda',
 'ebit',
 'nopi',
 'spi',
 'pi',
 'txp',
 'ni',
 'txfed',
 'txfo',
 'txt',
 'xint',
 'capx',
 'oancf',
 'dvt',
 'ob',
 'gdwlia',
 'gdwlip',
 'gwo',
 'rect',
 'act',
 'che',
 'ppegt',
 'invt',
 'at',
 'aco',
 'intan',
 'ao',
 'ppent',
 'gdwl',
 'fatb',
 'fatl',
 'lct',
 'dlc',
 'dltt',
 'lt',
 'dm',
 'dcvt',
 'cshrc',
 'dcpstk',
 'pstk',
 'ap',
 'lco',
 'lo',
 'drc',
 'drlt',
 'txdi',
 'ceq',
 'scstkc',
 'emp',
 'csho',
 'prcc_f']

(536477, 65)

In [4]:
# Get the Fundamental Quarterly data using SQL

fundq = db.raw_sql("""
    SELECT c.gvkey, c.sic,
           f.datadate, f.fyearq, f.fqtr,
           f.ibq, f.saleq, f.txtq, f.revtq, f.cogsq, f.xsgaq,
           f.atq, f.actq, f.cheq, f.lctq, f.dlcq, f.ppentq, f.prccq, f.cshoq,
           f.seqq, f.pstkq, f.ltq, f.pstkrq, f.rdq, f.ceqq
    FROM comp.fundq AS f
    JOIN comp.company AS c
      ON f.gvkey = c.gvkey
    WHERE f.datadate >= '1975-01-01'
      AND f.indfmt = 'INDL'
      AND f.datafmt = 'STD'
      AND f.popsrc = 'D'
      AND f.consol = 'C'
""").reset_index(drop=True)

fundq_raw = fundq.copy()

In [5]:
display(fundq.head(10))
display(list(fundq.columns))
display(fundq.shape)

,gvkey,sic,datadate,fyearq,fqtr,ibq,saleq,txtq,revtq,cogsq,...,dlcq,ppentq,prccq,cshoq,seqq,pstkq,ltq,pstkrq,rdq,ceqq
0,001000,3089,1975-03-31,1975,1,0.19,11.295,0.168,11.295,<NA>,...,0.8,<NA>,3.0,2.092,10.307,0.414,<NA>,<NA>,1975-05-02,9.893
1,001000,3089,1975-06-30,1975,2,0.875,14.441,0.777,14.441,<NA>,...,0.0,<NA>,2.875,2.092,11.182,0.414,<NA>,<NA>,1975-07-16,10.768
2,001000,3089,1975-09-30,1975,3,0.613,13.122,0.543,13.122,<NA>,...,0.0,<NA>,4.0,2.094,11.8,0.414,<NA>,<NA>,1975-10-09,11.386
3,001000,3089,1975-12-31,1975,4,0.606,12.334,0.452,12.334,35.237,...,1.251,7.263,4.375,2.098,12.309,2.069,11.596,<NA>,1976-03-04,10.24
4,001000,3089,1976-03-31,1976,1,0.726,16.226,0.569,16.226,11.138,...,1.674,12.059,6.75,2.096,13.062,2.069,19.469,<NA>,1976-04-20,10.993
5,001000,3089,1976-06-30,1976,2,1.603,19.637,1.26,19.637,12.981,...,1.257,12.693,7.25,2.163,14.87,2.069,20.287,<NA>,1976-07-14,12.801
6,001000,3089,1976-09-30,1976,3,0.561,14.733,0.441,14.733,10.722,...,1.507,12.93,7.125,2.172,15.501,2.069,19.322,<NA>,1976-10-18,13.432
7,001000,3089,1976-12-31,1976,4,0.544,15.818,0.47,15.818,12.034,...,0.35,16.44,5.75,2.207,15.675,1.569,22.911,<NA>,1977-03-01,14.106
8,001000,3089,1977-03-31,1977,1,0.563,18.014,0.48,18.014,13.206,...,0.35,16.665,6.25,2.209,16.248,1.569,25.861,<NA>,1977-05-03,14.679
9,001000,3089,1977-06-30,1977,2,0.975,21.203,<NA>,21.203,15.286,...,0.35,16.438,6.5,2.223,17.271,1.569,24.822,<NA>,1977-07-21,15.702


['gvkey',
 'sic',
 'datadate',
 'fyearq',
 'fqtr',
 'ibq',
 'saleq',
 'txtq',
 'revtq',
 'cogsq',
 'xsgaq',
 'atq',
 'actq',
 'cheq',
 'lctq',
 'dlcq',
 'ppentq',
 'prccq',
 'cshoq',
 'seqq',
 'pstkq',
 'ltq',
 'pstkrq',
 'rdq',
 'ceqq']

(2034725, 25)

In [6]:
# Get the Monthly CRSP Stock File using SQL

msf = db.raw_sql("""
                 SELECT permno, mthcaldt, mthret, mthprc, shrout, mthvol,  primaryexch, sharetype
                 FROM crsp.msf_v2
                 WHERE mthcaldt >= '1975-01-01'
                 """).reset_index(drop=True)

msf_raw = msf.copy()

In [76]:
# Get the CRSP DAILY Stock File using SQL

dsf = db.raw_sql('''
                 SELECT  permno, dlycaldt, dlyret, dlyprc, dlyvol, dlybid, dlyask, shrout
                 FROM crsp.dsf_v2
                 WHERE dlycaldt >= '1975-01-01'
                 ''').reset_index(drop=True)

dsf_raw = dsf.copy()

OperationalError: (psycopg2.OperationalError) could not receive data from server: Operation timed out
SSL SYSCALL error: Operation timed out

[SQL: 
                 SELECT  permno, dlycaldt, dlyret, dlyprc, dlyvol, dlybid, dlyask, shrout
                 FROM crsp.dsf_v2
                 WHERE dlycaldt >= '1975-01-01'
                 ]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
display(dsf.head(10))
display(list(dsf.columns))
display(dsf.shape)

In [ ]:
# Get the CRSP Daily Market Index using SQL

dsi = db.raw_sql("""
    SELECT date, vwretd
    FROM crsp.dsi
    WHERE date >= '1975-01-01'
""").reset_index(drop=True)

dsi_raw = dsi.copy()

In [ ]:
display(dsf.head(10))
display(list(dsi.columns))
display(dsi.shape)

In [8]:
# Get the CRSP Delisting Information using SQL ()

# Delisting returns capture the true gain/loss when a stock stops trading (e.g. bankruptcy), 
# so merging them in avoids survivorship bias that would otherwise make returns look artificially better.

dlret = db.raw_sql("""
                   SELECT permno, delistingdt, delret, delretmisstype, delactiontype, delstatustype, delreasontype
                   FROM crsp.stkdelists
                   """).reset_index(drop=True)

dlret_raw = dlret.copy()

In [9]:
display(dlret.head(10))
display(list(dlret.columns))
display(dlret.shape)

,permno,delistingdt,delret,delretmisstype,delactiontype,delstatustype,delreasontype
0,10000,1987-06-11,0.0,NA,GDR,VCL,INSC
1,10001,2017-08-03,0.0,NA,MER,FPAY,UNAV
2,10002,2013-02-15,0.010906,NA,MER,FPAY,UNAV
3,10003,1995-12-15,-0.003648,NA,MER,FPAY,UNAV
4,10004,1986-01-17,<NA>,DM,GDR,VCL,OFFRE
5,10005,1991-07-11,0.0,NA,GDR,VCL,INSC
6,10006,1984-06-28,0.011601,NA,MER,FPAY,UNAV
7,10007,1990-10-16,0.0,NA,GDR,VCL,INSC
8,10008,1988-11-21,0.0,NA,GDR,VCL,SHLD
9,10009,2000-11-03,0.003774,NA,MER,FPAY,UNAV


['permno',
 'delistingdt',
 'delret',
 'delretmisstype',
 'delactiontype',
 'delstatustype',
 'delreasontype']

(29833, 7)

In [10]:
# Get the CRSP/Compustat Merged link table using SQL

ccmlink = db.raw_sql("""
                     SELECT gvkey, lpermno, lpermco, linktype, linkprim, linkdt, linkenddt
                     FROM crsp.ccmxpf_lnkhist
                     WHERE linktype IN ('LC','LU')
                     AND linkprim IN ('P','C')
                     """).reset_index(drop=True)

ccmlink_raw = ccmlink.copy()

In [11]:
display(ccmlink.head(10))
display(list(ccmlink.columns))
display(ccmlink.shape)

,gvkey,lpermno,lpermco,linktype,linkprim,linkdt,linkenddt
0,001000,25881.0,23369.0,LU,P,1970-11-13,1978-06-30
1,001001,10015.0,6398.0,LU,P,1983-09-20,1986-07-31
2,001002,10023.0,22159.0,LC,C,1972-12-14,1973-06-05
3,001003,10031.0,6672.0,LU,C,1983-12-07,1989-08-16
4,001004,54594.0,20000.0,LU,P,1972-04-24,<NA>
5,001005,61903.0,11.0,LU,C,1973-01-31,1983-01-31
6,001007,10058.0,20.0,LU,C,1973-10-01,1979-01-30
7,001007,10058.0,20.0,LU,P,1979-01-31,1984-09-28
8,001008,10066.0,6331.0,LC,P,1983-08-25,1987-02-26
9,001009,10074.0,5271.0,LC,C,1982-01-18,1996-03-13


['gvkey', 'lpermno', 'lpermco', 'linktype', 'linkprim', 'linkdt', 'linkenddt']

(33324, 7)

In [12]:
# check if CRPS or Compustat tables names changed 

display(db.list_tables(library='crsp'))
display(db.list_tables(library='comp'))

['acti',
 'asia',
 'asib',
 'asic',
 'asio',
 'asix',
 'bmdebt',
 'bmheader',
 'bmpaymts',
 'bmquotes',
 'bmyield',
 'bndprt06',
 'bndprt12',
 'bxcalind',
 'bxdlyind',
 'bxmthind',
 'bxquotes',
 'bxyield',
 'cap',
 'ccm_lookup',
 'ccm_qvards',
 'ccmxpf_linktable',
 'ccmxpf_lnkhist',
 'ccmxpf_lnkrng',
 'ccmxpf_lnkused',
 'comphead',
 'comphist',
 'compmaster',
 'contact_info',
 'core',
 'crsp_cik_map',
 'crsp_daily_data',
 'crsp_header',
 'crsp_monthly_data',
 'crsp_names',
 'crsp_portno_map',
 'crsp_ziman_daily_index',
 'crsp_ziman_monthly_index',
 'cs20yr',
 'cs5yr',
 'cs90d',
 'cst_hist',
 'daily_nav',
 'daily_nav_ret',
 'daily_returns',
 'dividends',
 'dport1',
 'dport2',
 'dport3',
 'dport4',
 'dport5',
 'dport6',
 'dport7',
 'dport8',
 'dport9',
 'dsbc',
 'dsbo',
 'dse',
 'dse62',
 'dse62delist',
 'dse62dist',
 'dse62exchdates',
 'dse62names',
 'dse62nasdin',
 'dse62shares',
 'dseall',
 'dseall62',
 'dsedelist',
 'dsedist',
 'dseexchdates',
 'dsenames',
 'dsenasdin',
 'dseshares',

['aco_amda',
 'aco_imda',
 'aco_indfnta',
 'aco_indfntq',
 'aco_indfntytd',
 'aco_indsta',
 'aco_indstq',
 'aco_indstytd',
 'aco_notesa',
 'aco_notesq',
 'aco_notessa',
 'aco_notesytd',
 'aco_pnfnda',
 'aco_pnfndq',
 'aco_pnfndytd',
 'aco_pnfnta',
 'aco_pnfntq',
 'aco_pnfntytd',
 'aco_transa',
 'aco_transq',
 'aco_transsa',
 'aco_transytd',
 'adsprate',
 'asec_amda',
 'asec_imda',
 'asec_notesa',
 'asec_notesq',
 'asec_transa',
 'asec_transq',
 'bank_aacctchg',
 'bank_adesind',
 'bank_afnd1',
 'bank_afnd2',
 'bank_afnddc1',
 'bank_afnddc2',
 'bank_afntind',
 'bank_funda',
 'bank_funda_fncd',
 'bank_fundq',
 'bank_fundq_fncd',
 'bank_iacctchg',
 'bank_idesind',
 'bank_ifndq',
 'bank_ifndytd',
 'bank_ifntq',
 'bank_ifntytd',
 'bank_names',
 'bank_namesq',
 'chars',
 'co_aacctchg',
 'co_aaudit',
 'co_acthist',
 'co_adesind',
 'co_adjfact',
 'co_afnd1',
 'co_afnd2',
 'co_afnddc1',
 'co_afnddc2',
 'co_afntind1',
 'co_afntind2',
 'co_ainvval',
 'co_amkt',
 'co_busdescl',
 'co_cotype',
 'co_f

## 3) Merging and cleaning dataset

In [13]:
# resets to pristine if needed without rerunning SQL

# funda = funda_raw.copy().reset_index(drop=True)

In [14]:
# Cleaning funda

funda['datadate'] = pd.to_datetime(funda['datadate'])

funda['sic2'] = funda['sic'].str[:2]

funda['prcc_f'] = funda['prcc_f'].abs()

funda['mve_f'] = funda['csho'] * funda['prcc_f']

funda = funda.dropna(subset = ['at','prcc_f','ni'])

funda = funda.drop_duplicates(subset = ['gvkey','datadate'])

funda = funda.sort_values(['gvkey','datadate'])
funda['age'] = funda.groupby('gvkey').cumcount() + 1 # cumcount() gives each row within a group a sequential number starting at 0

funda['xint0'] = funda['xint'].fillna(0) # will not use xint

funda['xsga0'] = funda['xsga'].fillna(0) # will not use xsga

funda['dr'] = np.where(funda['drc'].notna() & funda['drlt'].notna(), funda['drc'] + funda['drlt'], 
              np.where(funda['drc'].notna(), funda['drc'],
              np.where(funda['drlt'].notna(), funda['drlt'], np.nan))) # can't add NA to numbers so need that
                                                                       # np.where is like if, then, dont forgot the last NA
                                                                       # we don't use the drlt and drc
                                                                       # dr is total deferred revenue (LT and Current)

# rebuild convertible debt (dc) - Compustat reports this inconsistently across companies — some break it out cleanly, others lump it together with convertible preferred stock.)

cond1 = funda['dcvt'].isna() & funda['dcpstk'].notna() & funda['pstk'].notna() & (funda['dcpstk'] > funda['pstk']) # True only if those 4 conditions hold
cond2 = funda['dcvt'].isna() & funda['dcpstk'].notna() & funda['pstk'].isna() # True only if those 3 conditions hold 
funda['dc'] = np.nan # create the dc columns (convertible debt) with NAs
funda.loc[cond1, 'dc'] = funda['dcpstk'] - funda['pstk'] # if condition 1 is TRUE, then do the substraction
funda.loc[cond2, 'dc'] = funda['dcpstk']  # if condition 2 is TRUE, get the raw dcpstk
funda['dc'] = funda['dc'].fillna(funda['dcvt']) # if still NA (not condition 1 neither condition 2), put the original convertible debt (dcvt)


In [15]:
# Resets to pristine if needed without rerunning SQL

# ccmlink = ccmlink_raw.copy().reset_index(drop=True)

In [16]:
# Merge gvkey and permno

ccmlink['linkdt'] = pd.to_datetime(ccmlink['linkdt'])
ccmlink['linkenddt'] = pd.to_datetime(ccmlink['linkenddt'])

ccmlink['linkenddt'] = ccmlink['linkenddt'].fillna(pd.Timestamp.today().normalize()) # from TidyFinance habit
ccmlink['linkdt'] = ccmlink['linkdt'].fillna(pd.Timestamp('1900-01-01'))

merged = funda.merge(ccmlink, on='gvkey', how='left')

valid = (merged['linkdt'] <= merged['datadate']) & (merged['datadate'] <= merged['linkenddt'])
funda = merged[valid].copy()

funda = funda.rename(columns={'lpermno': 'permno', 'lpermco': 'permco'})

funda = funda.dropna(subset=['permno'])

funda = funda.drop(columns=['linktype', 'linkprim', 'linkdt', 'linkenddt'])

In [41]:
# Merge msf to funda


funda['window_start'] = funda['datadate'] + pd.DateOffset(months=7)
funda['window_end'] = funda['datadate'] + pd.DateOffset(months=19)

msf['mthcaldt'] = pd.to_datetime(msf['mthcaldt'])
funda['permno'] = funda['permno'].astype('Int64')
msf['permno'] = msf['permno'].astype('Int64')

funda_sorted = funda.sort_values('window_start')
msf_sorted = msf.sort_values('mthcaldt')

monthly = pd.merge_asof(
    msf_sorted, funda_sorted,
    left_on='mthcaldt', right_on='window_start',
    by='permno', direction='backward'
)
monthly = monthly[monthly['mthcaldt'] < monthly['window_end'] + pd.DateOffset(months=1)]

monthly = monthly[(monthly['primaryexch'].isin(['N', 'A', 'Q'])) & (monthly['sharetype'] == 'NS')]

monthly.sort_values(by=['permno','mthcaldt'])

# The idea: a company's annual report for fiscal year ending datadate isn't actually known to investors until some months later (filing delay), so you shouldn't use it to predict returns starting right at datadate — that would be look-ahead bias (using information before it was public).

,permno,mthcaldt,mthret,mthprc,shrout,mthvol,primaryexch,sharetype,gvkey,cik,...,sic2,mve_f,age,xint0,xsga0,dr,dc,permco,window_start,window_end
852542,10000,1987-06-30,0.0,0.21875,3893,6722.0,Q,NS,013007,<NA>,...,39,2.88225,1.0,0.068,1.1,NaN,0.0,7952.0,1987-05-31,1988-05-31
811279,10001,1987-01-30,-0.035714,6.75,991,39895.0,Q,NS,012994,0000043350,...,49,5.91,1.0,0.332,0.0,NaN,0.0,7953.0,1987-01-30,1988-01-30
819603,10001,1987-02-27,-0.074074,6.25,991,36527.0,Q,NS,012994,0000043350,...,49,5.91,1.0,0.332,0.0,NaN,0.0,7953.0,1987-01-30,1988-01-30
830961,10001,1987-03-31,0.037486,6.375,991,21635.0,Q,NS,012994,0000043350,...,49,5.91,1.0,0.332,0.0,NaN,0.0,7953.0,1987-01-30,1988-01-30
836605,10001,1987-04-30,-0.039216,6.125,991,18776.0,Q,NS,012994,0000043350,...,49,5.91,1.0,0.332,0.0,NaN,0.0,7953.0,1987-01-30,1988-01-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4381967,93436,2025-08-29,0.083044,333.87,3225449,1598392667.0,Q,NS,184996,0001318605,...,37,1298749.44,15.0,350.0,9690.0,7478.0,0.0,53453.0,2025-07-31,2026-07-31
4394403,93436,2025-09-30,0.332015,444.72,3324000,1966930942.0,Q,NS,184996,0001318605,...,37,1298749.44,15.0,350.0,9690.0,7478.0,0.0,53453.0,2025-07-31,2026-07-31
4403965,93436,2025-10-31,0.026624,456.56,3325819,2024342219.0,Q,NS,184996,0001318605,...,37,1298749.44,15.0,350.0,9690.0,7478.0,0.0,53453.0,2025-07-31,2026-07-31
4418822,93436,2025-11-28,-0.057802,430.17,3325819,1609832217.0,Q,NS,184996,0001318605,...,37,1298749.44,15.0,350.0,9690.0,7478.0,0.0,53453.0,2025-07-31,2026-07-31


In [42]:
# Merge delisting info onto monthly, matched by permno + month

dlret['delistingdt'] = pd.to_datetime(dlret['delistingdt'])
dlret['permno'] = dlret['permno'].astype('Int64')

monthly['_ym'] = monthly['mthcaldt'].dt.to_period('M')
dlret['_ym'] = dlret['delistingdt'].dt.to_period('M')

monthly = monthly.merge(
    dlret[['permno', '_ym', 'delret', 'delactiontype', 'delreasontype']],
    on=['permno', '_ym'], how='left'
)
monthly = monthly.drop(columns='_ym')

# Identify "bad" (financial distress) delistings
bad_delist = (
    (monthly['delactiontype'] == 'GLI') |
    (monthly['delreasontype'].isin(['BKPY', 'INSC', 'INSF', 'DELQ', 'EQRQ', 'FING', 'LP', 'VIO']))
)

# Impute missing delret for bad delistings, based on exchange
monthly.loc[monthly['delret'].isna() & bad_delist & monthly['primaryexch'].isin(['N', 'A']), 'delret'] = -0.35
monthly.loc[monthly['delret'].isna() & bad_delist & (monthly['primaryexch'] == 'Q'), 'delret'] = -0.55

# Cap delret at -1 (can't lose more than 100%)
monthly.loc[monthly['delret'] < -1, 'delret'] = -1

# Fill any remaining missing delret with 0 (no delisting event, or not a bad one)
monthly['delret'] = monthly['delret'].fillna(0)

# Combine into the final return
monthly['ret_adj'] = monthly['mthret'] + monthly['delret']
monthly.loc[monthly['mthret'].isna() & (monthly['delret'] != 0), 'ret_adj'] = monthly['delret']

# Now safe to drop exchange columns
monthly = monthly.drop(columns=['primaryexch', 'sharetype'])

In [ ]:
# fundq

fundq = fundq.drop_duplicates(subset=['gvkey','datadate'])

fundq['datadate'] = pd.to_datetime(fundq['datadate'])

fundq['pstk'] = np.where(fundq['pstkrq'].notna(), fundq['pstkrq'], fundq['pstkq'])

fundq['scal'] = np.where(fundq['seqq'].notna(), fundq['seqq'],
                np.where(fundq['ceqq'].notna() & fundq['pstk'].notna(), fundq['ceqq'] + fundq['pstk'], fundq['atq'] - fundq['ltq']))

fundq['rdq'] = pd.to_datetime(fundq['rdq'])

fundq = fundq.sort_values(['gvkey', 'rdq'])

In [ ]:
# Link fundq to permno via CCM, same pattern as funda

merged_q = fundq.merge(ccmlink_raw[['gvkey', 'lpermno', 'linkdt', 'linkenddt']], on='gvkey', how='left')
merged_q['linkdt'] = pd.to_datetime(merged_q['linkdt']).fillna(pd.Timestamp('1900-01-01'))
merged_q['linkenddt'] = pd.to_datetime(merged_q['linkenddt']).fillna(pd.Timestamp.today().normalize())

valid_q = (merged_q['linkdt'] <= merged_q['datadate']) & (merged_q['datadate'] <= merged_q['linkenddt'])
fundq = merged_q[valid_q].copy()
fundq = fundq.rename(columns={'lpermno': 'permno'})
fundq = fundq.dropna(subset=['permno'])
fundq['permno'] = fundq['permno'].astype('Int64')
fundq = fundq.drop(columns=['linkdt', 'linkenddt'])

In [ ]:
# Build eamonth flag: is this calendar month the earnings-announcement month for this stock?

eamonth_lookup = fundq.dropna(subset=['permno', 'rdq'])[['permno', 'rdq']].drop_duplicates()
eamonth_lookup['ea_year'] = eamonth_lookup['rdq'].dt.year
eamonth_lookup['ea_month'] = eamonth_lookup['rdq'].dt.month
eamonth_lookup['eamonth'] = 1

monthly['ea_year'] = monthly['mthcaldt'].dt.year
monthly['ea_month'] = monthly['mthcaldt'].dt.month
monthly = monthly.merge(
    eamonth_lookup[['permno', 'ea_year', 'ea_month', 'eamonth']].drop_duplicates(subset=['permno', 'ea_year', 'ea_month']),
    on=['permno', 'ea_year', 'ea_month'], how='left'
)
monthly['eamonth'] = monthly['eamonth'].fillna(0)
monthly = monthly.drop(columns=['ea_year', 'ea_month'])

In [ ]:
# Merge fundq's quarterly data into monthly, using SAS's fixed 5-10 month lag window on datadate

monthly['q_window_end'] = monthly['mthcaldt'] - pd.DateOffset(months=5)
monthly['q_window_end'] = monthly['q_window_end'].values.astype('datetime64[M]')  # truncate to month start, matches SAS's 'beg'
monthly['q_window_start'] = monthly['mthcaldt'] - pd.DateOffset(months=10)

fundq_sorted = fundq.sort_values('datadate')
monthly_sorted = monthly.sort_values('q_window_end')

monthly = pd.merge_asof(
    monthly_sorted, fundq_sorted,
    left_on='q_window_end', right_on='datadate',
    by='permno', direction='backward', suffixes=('', '_q')
)
monthly = monthly[monthly['datadate_q'] >= monthly['q_window_start']]
monthly = monthly.drop(columns=['q_window_start', 'q_window_end'])

In [ ]:
display(dsf['dlycaldt'].dtype)

In [ ]:
# Sanity checks

# one permno per gvkey
# one permno per fyear/datadate
# one gvkey per fyear/datadate
# check NAs

#funda['gvkey'] = funda['gvkey'].astype('Int64')
#funda['cik'] = funda['cik'].astype('Int64')
#funda['permco'] = funda['permco'].astype('Int64')

In [ ]:
## 4) Variable creation

df = f

# acc

at_lag = df.groupby('gvkey')['at'].shift(1)
act_lag = df.groupby('gvkey')['act'].shift(1)
che_lag = df.groupby('gvkey')['che'].shift(1)
lct_lag = df.groupby('gvkey')['lct'].shift(1)
dlc_lag = df.groupby('gvkey')['dlc'].shift(1)
txp_lag = df.groupby('gvkey')['txp'].shift(1)

avg_at = (df['at'] + at_lag) / 2

df['acc'] = np.where(
    df['oancf'].notna(),
    (df['ib'] - df['oancf']) / avg_at,
    ((df['act'] - act_lag - (df['che'] - che_lag)) -
     ((df['lct'] - lct_lag) - (df['dlc'] - dlc_lag) - (df['txp'] - txp_lag) - funda['dp'])
    ) / avg_at
)

# absacc

df['absacc'] = df['acc'].abs()

# aeavol


